# Factor Combinations with Size
This notebook calculates value-weighted returns for portfolios formed by combining Size (Big/Small) with other factors (Value, Momentum, Profitability, Investment, Asset Turnover, etc). It plots cumulative returns for 10-year and 25-year horizons.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load data
file_path = 'Data/Updated_Factor_Data/total_universe/21_stock_level_monthly.csv'
df = pd.read_csv(file_path)

# Convert Month to datetime
df['Month'] = pd.to_datetime(df['Month'])

# Ensure numeric types
df['monthly_return'] = pd.to_numeric(df['monthly_return'], errors='coerce')
df['prev_mktcap'] = pd.to_numeric(df['prev_mktcap'], errors='coerce')

# Drop rows with missing returns or market caps
df = df.dropna(subset=['monthly_return', 'prev_mktcap'])

# Helper function to calculate Value-Weighted returns for a given factor
def calculate_vw_returns(df, factor_col):
    # Filter out rows missing size or the specific factor label
    valid_df = df.dropna(subset=['Size_Label_Monthly', factor_col]).copy()
    
    # Create a combined Size-Factor portfolio label
    valid_df['Portfolio'] = valid_df['Size_Label_Monthly'] + '_' + valid_df[factor_col]
    
    # Calculate weights within each month-portfolio
    valid_df['Total_Cap'] = valid_df.groupby(['Month', 'Portfolio'])['prev_mktcap'].transform('sum')
    valid_df['Weight'] = valid_df['prev_mktcap'] / valid_df['Total_Cap']
    
    # Calculate weighted return
    valid_df['VW_Return'] = valid_df['Weight'] * valid_df['monthly_return']
    
    # Sum weighted returns to get portfolio return
    port_returns = valid_df.groupby(['Month', 'Portfolio'])['VW_Return'].sum().unstack()
    return port_returns


In [ ]:
def plot_returns(returns, title, years=10):
    # Filter data to the last X years based on the max date in the dataset
    max_date = returns.index.max()
    start_date = max_date - pd.DateOffset(years=years)
    period_returns = returns[returns.index >= start_date].copy()
    
    # Calculate cumulative returns: (1 + r).cumprod()
    cum_returns = (1 + period_returns).cumprod()
    
    # Plot
    ax = cum_returns.plot(linewidth=2)
    plt.title(f'{title} - {years} Year Cumulative Returns ({start_date.strftime("%Y-%m")} to {max_date.strftime("%Y-%m")})')
    plt.xlabel('Date')
    plt.ylabel('Growth of 1 Unit')
    plt.legend(title='Portfolio', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

def analyze_factor(df, factor_col, factor_name, labels_mapping):
    print(f"Analyzing Size & {factor_name}")
    # Calculate returns
    returns = calculate_vw_returns(df, factor_col)
    
    # Rename columns using the mapping for better readability
    # e.g., 'S_W' -> 'Small Winner', 'B_W' -> 'Big Winner'
    new_cols = {}
    for col in returns.columns:
        if '_' in col:
            size, fac = col.split('_')
            size_name = 'Small' if size == 'S' else 'Big' if size == 'B' else size
            fac_name = labels_mapping.get(fac, fac)
            new_cols[col] = f"{size_name} {fac_name}"
            
    returns = returns.rename(columns=new_cols)
    
    # Fill missing returns with 0 for cumulative product
    returns = returns.fillna(0)
    
    # Plot 10 Years
    plot_returns(returns, f'Size & {factor_name}', years=10)
    
    # Plot 25 Years
    plot_returns(returns, f'Size & {factor_name}', years=25)


## 1. Size and Momentum
Momentum Labels: W (Winner), N (Neutral), L (Loser)

In [ ]:
mom_labels = {'W': 'Winner', 'N': 'Neutral', 'L': 'Loser'}
analyze_factor(df, 'MOM_Label', 'Momentum', mom_labels)


## 2. Size and Value (Book-to-Market)
Value Labels: V (Value), N (Neutral), G (Growth)

In [ ]:
value_labels = {'V': 'Value', 'N': 'Neutral', 'G': 'Growth'}
analyze_factor(df, 'BM_Label', 'Value', value_labels)


## 3. Size and Profitability
Profitability Labels: R (Robust), N (Neutral), W (Weak)

In [ ]:
profit_labels = {'R': 'Robust', 'N': 'Neutral', 'W': 'Weak'}
analyze_factor(df, 'OP_Label', 'Profitability', profit_labels)


## 4. Size and Investment
Investment Labels: C (Conservative), N (Neutral), A (Aggressive)

In [ ]:
inv_labels = {'C': 'Conservative', 'N': 'Neutral', 'A': 'Aggressive'}
analyze_factor(df, 'INV_Label', 'Investment', inv_labels)


## 5. Size and Asset Turnover
Asset Turnover Labels: H (High), N (Neutral), L (Low)

In [ ]:
at_labels = {'H': 'High', 'N': 'Neutral', 'L': 'Low'}
analyze_factor(df, 'AT_Label', 'Asset Turnover', at_labels)


## 6. Size and Sales Growth
Sales Growth Labels: H (High), N (Neutral), L (Low)

In [ ]:
sg_labels = {'H': 'High', 'N': 'Neutral', 'L': 'Low'}
analyze_factor(df, 'SG_Label', 'Sales Growth', sg_labels)


## 7. Size and Accruals
Accruals Labels: C (Conservative), N (Neutral), A (Aggressive)

In [ ]:
acc_labels = {'C': 'Conservative', 'N': 'Neutral', 'A': 'Aggressive'}
analyze_factor(df, 'ACC_Label', 'Accruals', acc_labels)


## 8. Size and Volatility
Volatility Labels: L (Low), N (Neutral), H (High)

In [ ]:
vol_labels = {'L': 'Low', 'N': 'Neutral', 'H': 'High'}
analyze_factor(df, 'VOL_Label', 'Volatility', vol_labels)


## 9. Size and Short-Term Reversal
Short-Term Reversal Labels: L (Loser), N (Neutral), H (Winner)

In [ ]:
str_labels = {'L': 'Loser', 'N': 'Neutral', 'H': 'Winner'}
analyze_factor(df, 'STR_Label', 'Short-Term Reversal', str_labels)
